# MAPS (authors' code) on multiple datasets: 5-fold F1 evaluation
This notebook runs the **official MAPS model code** (Shaban et al., Nat Commun 2024; repo `mahmoodlab/MAPS`, commit `72d4bc8`) on your cell tables.

**What is the authors' exact code here**
- The files `datasets.py`, `networks.py`, `trainer.py`, `predictor.py` are copied **verbatim** from the repo (cells 3-7). The model, normalization, sampler, loss, early stopping and checkpointing are all theirs.
- Training settings default to the values in the authors' `CRC_CODEX.sh` (batch 128, lr 0.001, dropout 0.25, 500/100/50 epochs/min-epochs/patience, seed 7325111).
- Cell 8 applies **2 tiny compatibility patches** (see that cell). Without them the authors' code crashes on current NumPy/PyTorch.

**What is not theirs (they never released it for your datasets)**
- Turning each dataset's table into MAPS format (markers + cell size + `cell_label`) and the 5-fold split. I mirror their approach: folds are made **by patient/region/spot** using their size-balancing rule, and the training part is split 80/20 into train/valid (like their `cHL_CODEX.py`), so the test fold is never used for early stopping.

**How to use:** attach your dataset to the notebook as Kaggle input, then **Run All**. The notebook finds the data, detects the columns, trains the 5 folds and saves the results. Nothing needs editing unless the *Check* cell shows a wrong column (then fill `OVERRIDES` in cell 5). Finished folds are skipped if you re-run after a crash/timeout.

## 1. Imports

In [ ]:
import os, sys, gc, glob, re, json, time, shutil, warnings, contextlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
warnings.filterwarnings("ignore")
try:
    import anndata  # only needed for .h5ad files
except ImportError:
    try:
        import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anndata"], check=True); import anndata
    except Exception:
        print("anndata not available (fine unless you have .h5ad files; needs Internet ON)")
import torch
print("numpy", np.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Authors' code (verbatim copies from the MAPS repo)

In [ ]:
import os
os.makedirs('maps_src/maps/cell_phenotyping', exist_ok=True)

In [ ]:
%%writefile maps_src/maps/cell_phenotyping/__init__.py
from .trainer import Trainer
from .predictor import Predictor

__all__ = [
    'Trainer',
    'Predictor',
]


In [ ]:
%%writefile maps_src/maps/cell_phenotyping/datasets.py
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader, SequentialSampler, WeightedRandomSampler


class CellExpressionCSV(Dataset):
    """
    A custom dataset class to read in and process cell expression data in CSV format.

    Args:
        csv_path (str): The path to the CSV file containing cell expression data.
        is_train (bool, optional): A flag indicating whether the data is used for training or not. Defaults to True.
        mean (numpy.ndarray, optional): The mean of the training data. Required if is_train is False. Defaults to None.
        std (numpy.ndarray, optional): The standard deviation of the training data. Required if is_train is False. Defaults to None.

    Attributes:
        x (numpy.ndarray): The cell expression data.
        y (numpy.ndarray): The cell labels, if present in the CSV file.
        is_train (bool): A flag indicating whether the data is used for training or not.
        mean (numpy.ndarray): The mean of the training data.
        std (numpy.ndarray): The standard deviation of the training data.
    """

    def __init__(self, csv_path, is_train=True, mean=None, std=None):
        self.x = None
        self.y = None
        self.is_train = is_train
        self.mean = mean
        self.std = std

        # Read the CSV file and extract cell expression data and labels
        df = pd.read_csv(csv_path)
        if 'cell_label' in df.columns.tolist():
            self.y = df['cell_label'].to_numpy()
            df = df.drop(columns=['cell_label'])
        self.x = df.to_numpy()

        # Normalize the cell expression data
        if is_train:
            if (self.mean is None) or (self.std is None):
                self.mean = np.mean(self.x, axis=0)
                self.std = np.std(self.x, axis=0)
                self.x = (self.x - self.mean) / self.std
            else:
                raise ValueError('Mean and/or std is None.')
        else:
            if (self.mean is not None) and (self.std is not None):
                self.x = (self.x - self.mean) / self.std

    def __len__(self):
        """
        Return the number of samples in the dataset.

        Returns:
            int: The number of samples in the dataset.
        """
        return self.x.shape[0]

    def __getitem__(self, idx):
        """
        Return the features and ground truth for a given index.

        Parameters:
        idx (int): Index of the data.

        Returns:
        feature (numpy.ndarray): Features of the data.
        gt (int): Ground truth label of the data. -1 if not available.
        """
        feature = self.x[idx] / 255.0
        if self.y is not None:
            gt = int(self.y[idx])
        else:
            gt = -1

        return feature, gt

    @staticmethod
    def get_data_loader(dataset, batch_size=4, is_train=False, num_workers=4):
        """
        Given a PyTorch dataset, this function returns a PyTorch dataloader.

        Parameters:
        - dataset (Dataset): PyTorch dataset to be loaded into a dataloader.
        - batch_size (int, optional): Number of samples in each batch. Defaults to 4.
        - is_train (bool, optional): Flag indicating whether the data is for training or not. Defaults to False.
        - num_workers (int, optional): Number of worker threads for data loading. Defaults to 4.

        Returns:
        - DataLoader: PyTorch dataloader containing the given dataset.
        """
        # If the data is for training, use WeightedRandomSampler to create a sampler for the data
        if is_train:
            labels = dataset.y.tolist()
            n = float(len(dataset))
            weight = [0] * int(n)
            weight_per_class = [n / labels.count(c) for c in np.unique(labels)]
            label_unique = np.unique(labels).tolist()
            for idx in range(len(dataset)):
                label = labels[idx]
                weight[idx] = weight_per_class[label_unique.index(label)]

            loader = DataLoader(dataset, batch_size=batch_size, sampler=WeightedRandomSampler(weight, len(weight)),
                                drop_last=True, num_workers=num_workers)
        else:
            loader = DataLoader(dataset, batch_size=batch_size, sampler=SequentialSampler(dataset), drop_last=False,
                                num_workers=num_workers)

        return loader


In [ ]:
%%writefile maps_src/maps/cell_phenotyping/networks.py
import torch
import torch.nn as nn


class MLP(nn.Module):
    """
    Multi-layer Perceptron (MLP) - Feedforward Neural Network Model

    This model is composed of multiple fully connected (Linear) layers followed by non-linear activation functions
    and dropout layers to reduce overfitting. The final layer is the output layer, which returns logits and class
    probabilities.

    Arguments:
        input_dim (int): number of input features
        hidden_dim (int): number of hidden units in each fully connected layer
        num_classes (int): number of classes in the classification task
        dropout (float): dropout rate applied after each fully connected layer

    """
    def __init__(self, input_dim=47, hidden_dim=512, num_classes=12, dropout=0.10):
        super(MLP, self).__init__()  # Inherited from the parent class nn.Module
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout)
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, batch):
        """
        Forward Propagation of the MLP Model

        Arguments:
            batch (torch.Tensor): Input features of shape (batch_size, input_dim)

        Returns:
            logits (torch.Tensor): Logits of shape (batch_size, num_classes)
            probs (torch.Tensor): Class probabilities of shape (batch_size, num_classes)

        """
        features = self.fc(batch)
        logits = self.classifier(features)
        probs = torch.softmax(logits, dim=-1)
        return logits, probs


In [ ]:
%%writefile maps_src/maps/cell_phenotyping/trainer.py
from __future__ import print_function
import os
import sys
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

from .datasets import CellExpressionCSV
from .networks import MLP


class Trainer:
    """
    Class to train a cell phenotype classification model.

    Attributes:
    ----------
    model_checkpoint_path: str, optional
        Path to a pre-trained model checkpoint. Default is None.
    results_dir: str, optional
        Path to directory to store the training results. Default is './results/'.
    num_features: int, optional
        The number of features in the input data. Default is 47.
    num_classes: int, optional
        The number of classes in the classification task. Default is 12.
    batch_size: int, optional
        The batch size used for training and evaluation. Default is 128.
    learning_rate: float, optional
        The learning rate for the Adam optimizer. Default is 0.001.
    dropout: float, optional
        The dropout rate used in the MLP network. Default is 0.10.
    max_epochs: int, optional
        The maximum number of epochs to run the training. Default is 500.
    min_epochs: int, optional
        The minimum number of epochs to run the training. Default is 250.
    patience: int, optional
        The patience used in early stopping. Default is 100.
    seed: int, optional
        The seed used to initialize the random number generator. Default is 7325111.
    num_workers: int, optional
        The number of workers used in the data loading. Default is 4.
    verbose: int, optional
        The verbosity level. Default is 0.

    """

    def __init__(self, model_checkpoint_path=None, results_dir='./results/', num_features=47, num_classes=12,
                 batch_size=128, learning_rate=0.001, dropout=0.10, max_epochs=500,
                 min_epochs=250, patience=100, seed=7325111, num_workers=4, verbose=0):
        """
        Initialize the CellPhenotypeTrainer class.
        """
        self.model_checkpoint_path = model_checkpoint_path
        self.results_dir = results_dir
        self.num_features = num_features
        self.num_classes = num_classes
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.dropout = dropout
        self.max_epochs = max_epochs
        self.min_epochs = min_epochs
        self.patience = patience
        self.num_workers = num_workers
        self.seed = seed
        self.verbose = verbose

        self.model = None
        self.optimizer = None
        self.loss_fn = None
        self.counter = 0
        self.lowest_loss = np.Inf
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def set_seed(self):
        """
        Sets the seed for the random number generator.

        This function sets the seed for both the NumPy and PyTorch random number generators, ensuring that results are reproducible.

        Parameters:
        seed (int): The seed to be set for the random number generators.

        Returns:
        None
        """
        random.seed(self.seed)
        os.environ['PYTHONHASHSEED'] = str(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if str(self.device.type) == 'cuda':
            torch.cuda.manual_seed(self.seed)
            torch.cuda.manual_seed_all(self.seed)  # if you are using multi-GPU.
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    def init_data_loader(self, data_path, is_train=False, mean=None, std=None):
        """
        Initialize and return the data loader for the given data path.

        Parameters:
        - data_path (str): Path to the data file.
        - is_train (bool, optional): Whether to return the data loader for training or not. Default is False.
        - mean (list, optional): List of mean values to be used for normalization. Default is None.
        - std (list, optional): List of standard deviation values to be used for normalization. Default is None.

        Returns:
        - torch.utils.data.DataLoader: The data loader for the given data path.
        """
        dataset = CellExpressionCSV(data_path, is_train=is_train, mean=mean, std=std)
        return CellExpressionCSV.get_data_loader(dataset, batch_size=self.batch_size, is_train=is_train,
                                                 num_workers=self.num_workers)

    def init_model(self):
        """
        Initialize the Multi-layer Perceptron (MLP) model for cell expression classification.

        Parameters:
        None

        Returns:
        None

        """
        self.model = MLP(input_dim=self.num_features, hidden_dim=512, num_classes=self.num_classes,
                         dropout=self.dropout)
        self.model.to(self.device, dtype=torch.float64)

    def init_optimizer(self):
        """
        Initialize the optimizer for the model.

        Returns:
            None
        """
        self.optimizer = optim.Adam(filter(lambda p: p.requires_grad, self.model.parameters()), lr=self.learning_rate)

    def init_loss_function(self):
        """
        Initialize the loss function for the model.

        Returns:
            None
        """
        self.loss_fn = nn.CrossEntropyLoss()

    def save_model(self, mean, std):
        """
        This function saves the current state of the model along with mean and standard deviation of the train data.

        Parameters:
        mean (float): mean of the train data
        std (float): standard deviation of the train data

        Returns:
        None
        """
        os.makedirs(self.results_dir, exist_ok=True)  # Creating results directory if it is not there.
        save_dict = {'model_parameters': self.model.state_dict(), 'train_data_mean': mean, 'train_data_std': std}
        torch.save(save_dict, os.path.join(self.results_dir, 'best_checkpoint.pt'))
        self.model_checkpoint_path = os.path.join(self.results_dir, 'best_checkpoint.pt')

    def load_model(self):
        """
        This function loads the saved state of the model along with mean and standard deviation of the train data.

        Returns:
        mean (float): mean of the train data
        std (float): standard deviation of the train data
        """
        if self.model_checkpoint_path is None:
            raise ValueError('Model Checkpoint path is None.')
        else:
            ckpt_path = self.model_checkpoint_path
            loaded_dict = torch.load(ckpt_path, map_location=self.device)
            self.model.load_state_dict(loaded_dict['model_parameters'], strict=True)
        return loaded_dict['train_data_mean'], loaded_dict['train_data_std']

    def fit(self, train_dataset_path, valid_dataset_path):
        """
        Trains the model with the provided training dataset and validation dataset
        and saves the best model based on the lowest validation loss.
        The training progress, including loss, accuracy, and AUC, is recorded
        and saved in a CSV file in the "results" directory.

        Parameters:
        train_dataset_path (str): Path to the training dataset.
        valid_dataset_path (str): Path to the validation dataset.

        Returns:
        None
        """

        self.counter = 0
        self.lowest_loss = np.Inf
        self.set_seed()
        self.init_model()
        self.init_optimizer()
        self.init_loss_function()
        train_dl = self.init_data_loader(train_dataset_path, is_train=True)
        train_dataset_mean = train_dl.dataset.mean
        train_data_std = train_dl.dataset.std
        valid_dl = self.init_data_loader(valid_dataset_path, mean=train_dataset_mean, std=train_data_std)

        result_dict = {'train_loss': [], 'valid_loss': [],
                       'train_acc': [], 'valid_acc': [],
                       'train_auc': [], 'valid_auc': []}
        for epoch in range(self.max_epochs):
            start_time = time.time()

            # Train the model for one epoch
            train_loss, train_acc, train_auc = self.train_loop(train_dl)
            print('\rTrain Epoch: {}, train_loss: {:.4f}, train_acc: {:.4f}, train_auc: {:.4f}                 '.format(
                epoch, train_loss, train_acc, train_auc))
            # Record the training loss, accuracy, and AUC
            result_dict['train_loss'].append(train_loss)
            result_dict['train_acc'].append(train_acc)
            result_dict['train_auc'].append(train_auc)

            # Evaluate the model on the validation dataset
            valid_loss, valid_acc, valid_auc = self.valid_loop(valid_dl)
            print('\rValid Epoch: {}, valid_loss: {:.4f}, valid_acc: {:.4f}, valid_auc: {:.4f}                 '.format(
                epoch, valid_loss, valid_acc, valid_auc))
            # Record the validation loss, accuracy, and AUC
            result_dict['valid_loss'].append(valid_loss)
            result_dict['valid_acc'].append(valid_acc)
            result_dict['valid_auc'].append(valid_auc)

            # Save the model with the lowest validation loss
            if self.lowest_loss > valid_loss:
                print('--------------------Saving best model--------------------')
                self.save_model(train_dataset_mean, train_data_std)
                self.lowest_loss = valid_loss
                self.counter = 0
            else:
                self.counter += 1
                print('Loss is not decreased in last %d epochs' % self.counter)

            if (self.counter > self.patience) and (epoch >= self.min_epochs):
                break

            total_time = time.time() - start_time
            print('Time to process epoch({}): {:.4f} minutes                             \n'.format(epoch, total_time/60))

            pd.DataFrame.from_dict(result_dict).to_csv(os.path.join(self.results_dir, 'training_logs.csv'))

    def predict(self, test_data_path):
        """
        This function is used to make predictions on the test data.

        Parameters:
        test_data_path (str): path to the test data

        Returns:
        pred_labels (numpy.ndarray): List of predicted class labels
        pred_probs (numpy.ndarray): List of predicted class probabilities
        """
        self.set_seed()
        self.init_model()
        self.load_model()
        train_data_mean, train_data_std = self.load_model()

        data_loader = self.init_data_loader(test_data_path, mean=train_data_mean, std=train_data_std)

        pred_labels, pred_probs = self.test_loop(data_loader)

        return pred_labels, pred_probs

    def train_loop(self, data_loader):
        """
        Train the model using the training data provided in the `data_loader` argument.

        Parameters:
        data_loader (torch.utils.data.DataLoader): The PyTorch data loader containing the training data.

        Returns:
        - avg_loss: The average training loss for the epoch.
        - acc: The accuracy of the model on the training data.
        - auc: The area under the receiver operating characteristic curve (AUC) of the model on the training data.
        """
        total_loss = 0
        gt_labels = []
        pred_labels = []
        pred_probs = None
        batch_count = len(data_loader)
        self.model.train()
        for batch_idx, (features_batch, label_batch) in enumerate(data_loader):
            # Convert ground-truth labels to a list
            if len(gt_labels) == 0:
                gt_labels = label_batch.cpu().numpy().tolist()
            else:
                gt_labels.extend(label_batch.cpu().numpy().tolist())

            # Transfer features and labels to the GPU if one is available
            features_batch = features_batch.to(self.device)
            label_batch = label_batch.to(self.device)

            # Forward pass through the model
            logits, probs = self.model(features_batch)

            # Calculate the loss and backpropagate the gradients
            self.optimizer.zero_grad()
            loss = self.loss_fn(logits, label_batch)
            loss.backward()
            self.optimizer.step()

            # Accumulate the loss
            total_loss += loss.cpu().item()

            # Convert predicted labels and probabilities to lists
            p_labels = torch.argmax(probs, dim=1)
            p_labels = p_labels.detach().cpu().numpy().tolist()
            p_probs = probs.detach().cpu().numpy()

            # Accumulate the predicted labels and probabilities
            if len(pred_labels) == 0:
                pred_probs = p_probs
                pred_labels = p_labels
            else:
                pred_probs = np.concatenate((pred_probs, p_probs), axis=0)
                pred_labels.extend(p_labels)

            # Output the training progress
            sys.stdout.write('\rTraining Batch {}/{}, avg loss: {:.4f}'.format(
                batch_idx + 1, batch_count, total_loss / (batch_idx + 1)))

        # Compute accuracy and AUC of the model
        acc = accuracy_score(gt_labels, pred_labels)
        auc = roc_auc_score(gt_labels, pred_probs, multi_class='ovo', labels=[i for i in range(self.num_classes)])

        # Output the confusion matrix if verbosity is set to 1
        if self.verbose == 1:
            print('\nConfusion Matrix')
            conf = pd.DataFrame(confusion_matrix(gt_labels, pred_labels, labels=[i for i in range(self.num_classes)]),
                                index=['class_%d' % i for i in range(self.num_classes)],
                                columns=['class_%d' % i for i in range(self.num_classes)])
            print(conf)
        return total_loss/batch_count, acc, auc

    def valid_loop(self, data_loader):
        """
        The function to evaluate the model on the validation dataset.

        Parameters:
        - data_loader: PyTorch Dataloader for the validation dataset.

        Returns:
        - avg_loss: Average validation loss across all batches.
        - acc: Validation accuracy score.
        - auc: Validation area under the ROC curve score.
        """
        total_loss = 0
        gt_labels = []
        pred_labels = []
        pred_probs = None
        batch_count = len(data_loader)
        self.model.eval()
        with torch.no_grad():
            for batch_idx, (features_batch, label_batch) in enumerate(data_loader):
                # Store the ground truth labels
                if len(gt_labels) == 0:
                    gt_labels = label_batch.cpu().numpy().tolist()
                else:
                    gt_labels.extend(label_batch.cpu().numpy().tolist())

                # Move the input and label tensors to the GPU if specified
                features_batch = features_batch.to(self.device)
                label_batch = label_batch.to(self.device)

                # Get the model's logits and predicted probabilities for the current batch
                logits, probs = self.model(features_batch)

                # Calculate the loss for the current batch
                loss = self.loss_fn(logits, label_batch)

                # Add the batch loss to the running total loss
                total_loss += loss.cpu().item()

                # Get the predicted labels and probabilities
                p_labels = torch.argmax(probs, dim=1)
                p_labels = p_labels.detach().cpu().numpy().tolist()
                p_probs = probs.detach().cpu().numpy()

                # Store the predicted labels and probabilities
                if len(pred_labels) == 0:
                    pred_probs = p_probs
                    pred_labels = p_labels
                else:
                    pred_probs = np.concatenate((pred_probs, p_probs), axis=0)
                    pred_labels.extend(p_labels)

                # Print the progress to the console
                sys.stdout.write('\rValidation Batch {}/{}, avg loss: {:.4f}'.format(
                    batch_idx + 1, batch_count, total_loss / (batch_idx + 1)))

            # Calculate the accuracy and AUC scores
            acc = accuracy_score(gt_labels, pred_labels)
            auc = roc_auc_score(gt_labels, pred_probs, multi_class='ovo',
                                labels=[i for i in range(self.num_classes)])
            # If verbose is set to 1, print the confusion matrix
            if self.verbose == 1:
                print('\nConfusion Matrix')
                conf = pd.DataFrame(confusion_matrix(gt_labels, pred_labels, labels=[i for i in range(self.num_classes)]),
                    index=['class_%d' % i for i in range(self.num_classes)],
                    columns=['class_%d' % i for i in range(self.num_classes)])
                print(conf)
        return total_loss/batch_count, acc, auc

    def test_loop(self, data_loader):
        """
        Evaluates the model on the given data_loader.

        Parameters:
        data_loader (torch.utils.data.DataLoader): The DataLoader that contains the test data.

        Returns:
        pred_labels (numpy.ndarray): The predicted labels for the test data.
        pred_probs (numpy.ndarray): The predicted probabilities for each class for the test data.

        """
        # Initialize the arrays for storing the predicted labels and probabilities
        pred_labels = None
        pred_probs = None

        # Get the number of batches in the data_loader
        batch_count = len(data_loader)

        # Put the model in evaluation mode
        self.model.eval()

        # Turn off gradient computation to speed up the evaluation
        with torch.no_grad():
            # Loop over the batches in the data_loader
            for batch_idx, (features_batch, _) in enumerate(data_loader):
                # Move the features batch to the device (GPU/CPU) specified in the instance
                features_batch = features_batch.to(self.device)

                # Forward the features through the model to get the logits and predicted probabilities
                logits, probs = self.model(features_batch)

                # Convert the predicted probabilities from tensor to numpy array
                probs = probs.detach().cpu().numpy()

                # Get the predicted labels as the class with the maximum predicted probability
                labels = np.argmax(probs, axis=1)

                # Concatenate the predicted labels and probabilities from this batch to the accumulated results
                if pred_labels is None:
                    pred_labels = labels
                    pred_probs = probs
                else:
                    pred_labels = np.concatenate((pred_labels, labels), axis=0)
                    pred_probs = np.concatenate((pred_probs, probs), axis=0)

                # Print the progress of the evaluation in the form of 'Batch x/y'
                sys.stdout.write('\rBatch {}/{}            '.format(batch_idx + 1, batch_count))

        # Return the accumulated predicted labels and probabilities
        return pred_labels, pred_probs


In [ ]:
%%writefile maps_src/maps/cell_phenotyping/predictor.py
from __future__ import print_function
import os
import sys
import random
import numpy as np

import torch

from .datasets import CellExpressionCSV
from .networks import MLP
from .trainer import Trainer


class Predictor(Trainer):
    """
    Class to predict a cell phenotype using pretrained classification model.

    Attributes:
    ----------
    model_checkpoint_path: str, optional
        Path to a pre-trained model checkpoint. Default is None.
    results_dir: str, optional
        Path to directory to store the training results. Default is './results/'.
    num_features: int, optional
        The number of features in the input data. Default is 47.
    num_classes: int, optional
        The number of classes in the classification task. Default is 12.
    batch_size: int, optional
        The batch size used for training and evaluation. Default is 128.
    seed: int, optional
        The seed used to initialize the random number generator. Default is 7325111.
    num_workers: int, optional
        The number of workers used in the data loading. Default is 4.
    normalization : bool, optional
        Flag to indicate whether to normalize data before inference using pre-calculated mean and standard deviation 
        from training dataset. Default is True.
    verbose: int, optional
        The verbosity level. Default is 0.

    """

    def __init__(self, model_checkpoint_path=None, results_dir='./results/', num_features=47, num_classes=12,
                 batch_size=128, seed=7325111, num_workers=4, normalization=True, verbose=0):
        """
        Initialize the CellPhenotypeTrainer class.
        """
        super().__init__(model_checkpoint_path=model_checkpoint_path,
                         results_dir=results_dir,
                         num_features=num_features,
                         num_classes=num_classes,
                         batch_size=batch_size,
                         seed=seed,
                         num_workers=num_workers,
                         verbose=verbose)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.set_seed()
        self.init_model()
        self.load_model()
        if normalization:
            self.train_data_mean, self.train_data_std = self.load_model()
        else:
            self.train_data_mean, self.train_data_std = None, None

    def predict(self, test_data_path):
        """
        This function is used to make predictions on the test data.

        Parameters:
        test_data_path (str): path to the test data

        Returns:
        pred_labels (numpy.ndarray): List of predicted class labels
        pred_probs (numpy.ndarray): List of predicted class probabilities
        """

        data_loader = self.init_data_loader(test_data_path, mean=self.train_data_mean, std=self.train_data_std)

        pred_labels, pred_probs = self.test_loop(data_loader)

        return pred_labels, pred_probs


In [ ]:
%%writefile maps_src/maps/__init__.py
from . import cell_phenotyping

__all__ = [
    'cell_phenotyping',
]


## 3. Compatibility patches (the only edits to the authors' code)
1. `np.Inf` → `np.inf` in `trainer.py` (`np.Inf` was removed in NumPy 2.0, so the original crashes).
2. `torch.load(...)` → `torch.load(..., weights_only=False)` in `trainer.py` (PyTorch ≥ 2.6 refuses to load their checkpoint, which stores NumPy arrays, otherwise).

Neither changes the maths.

In [ ]:
p = "maps_src/maps/cell_phenotyping/trainer.py"
s = open(p).read()
n1, n2 = s.count("np.Inf"), s.count("torch.load(ckpt_path, map_location=self.device)")
assert n1 == 2 and n2 == 1, (n1, n2)
s = s.replace("np.Inf", "np.inf").replace("torch.load(ckpt_path, map_location=self.device)",
                                          "torch.load(ckpt_path, map_location=self.device, weights_only=False)")
open(p, "w").write(s)
sys.path.insert(0, os.path.abspath("maps_src"))
from maps.cell_phenotyping import Trainer, Predictor
print("authors' MAPS code loaded; patched np.Inf x2 and torch.load x1")

## 4. Files you attached (Kaggle)

In [ ]:
INPUT_ROOT = os.environ.get("MAPS_INPUT_ROOT") or ("/kaggle/input" if os.path.exists("/kaggle/input") else ".")
for root, dirs, files in os.walk(INPUT_ROOT):
    if root[len(INPUT_ROOT):].count(os.sep) > 3: continue
    for f in files[:25]:
        pth = os.path.join(root, f); print(f"{os.path.getsize(pth)/1e6:9.1f} MB  {pth}")
    if len(files) > 25: print(f"          ... {len(files)-25} more files in {root}")

## 5. Automatic dataset discovery (nothing to edit)
Looks through everything attached under `/kaggle/input`, groups table files by identical columns (so per-sample CSVs get stacked, metadata files get ignored), and picks the group that has a cell-type column. If the Schürch CRC CODEX table is attached (columns `ClusterName`, `spots` + the 34 named markers), the authors' own label mapping is applied automatically.

Only if the *Check* cell shows a wrong column: fill `OVERRIDES`, e.g. `OVERRIDES = dict(label_col="cell_type", group_col="patient_id")`. Other keys: `area_col`, `marker_cols`, `class_mapping`, `drop_labels`, `hp` (e.g. `hp=dict(max_epochs=100, min_epochs=50, patience=20)`).

In [ ]:
OVERRIDES = dict()   # leave empty unless the Check cell shows a wrong column

In [ ]:
# ---- global settings ----
OUT_ROOT = "/kaggle/working/maps_results" if os.path.exists("/kaggle/working") else "maps_results"
N_FOLDS = 5
MIN_CELLS_PER_CLASS = 100  # classes with fewer cells are dropped (MAPS' per-epoch AUC crashes if a class is missing from a split)
DROP_LABELS = ["", "nan", "none", "undefined", "unassigned", "unknown", "artifact", "dirt", "seg artifact"]  # case-insensitive
SAVE_PROBS = False         # True = also save per-cell class probabilities (big files)
DEFAULT_HP = dict(batch_size=128, learning_rate=0.001, dropout=0.25,      # = authors' CRC_CODEX.sh
                  max_epochs=500, min_epochs=100, patience=50, seed=7325111, num_workers=4)
SPLIT_SEED = 7325111       # authors' seed, used for the train/valid carve-out
os.makedirs(OUT_ROOT, exist_ok=True)

## 6. Helper functions

In [ ]:
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
LABEL_CAND = ["clustername","celltype","celltypename","cellphenotype","phenotype","cellannotation","annotation","cellclass",
              "majorcelltype","celltypemajor","metacluster","cellmetacluster","cluster","clusters","label","labels","cellsubtype"]
GROUP_CAND = ["patientid","patient","patients","donor","donorid","subject","subjectid","caseid","case","sampleid","sample",
              "samplename","slideid","slide","imageid","image","imagenumber","roi","roiid","region","regionid","spots","spot",
              "fov","tissueid","tissue","core"]
AREA_CAND  = ["cellsize","area","cellarea","size","nucleusarea","nucleussize","areapx"]
NONMARKER_RE = re.compile(r"(^|[^a-z])(x|y|z)([^a-z]|$)|centroid|coord|^index$|cell.?id|^id$|object.?id|label|fov|region|"
                          r"sample|patient|image|slide|core|roi|eccentric|solidity|perimeter|orientation|axis|extent|"
                          r"neighbo|cluster|annot|type|phenotype|tissue|donor|case|batch|unnamed|spot|tile|^__", re.I)
EXTS = (".csv", ".tsv", ".csv.gz", ".tsv.gz", ".parquet", ".h5ad")

# ---- authors' own mapping for the Schurch CRC CODEX table (copied from scripts/MAPS/data_preprocessing/CRC_CODEX.py) ----
CRC_CLASS_MAPPING = {'B cells': 'B cells', 'CD11b+CD68+ macrophages': 'Macrophages', 'CD11c+ DCs': 'CD11c+ DCs',
    'CD163+ macrophages': 'Macrophages', 'CD4+ T cells': 'CD4+ T cells', 'CD4+ T cells CD45RO+': 'CD4+ T cells',
    'CD4+ T cells GATA3+': 'CD4+ T cells', 'CD68+ macrophages': 'Macrophages', 'CD68+ macrophages GzmB+': 'Macrophages',
    'CD68+CD163+ macrophages': 'Macrophages', 'CD8+ T cells': 'CD8+ T cells', 'NK cells': 'NK cells', 'Tregs': 'Tregs',
    'granulocytes': 'Granulocytes', 'immune cells': 'Immune cells', 'plasma cells': 'Plasma cells',
    'smooth muscle': 'Smooth muscle', 'stroma': 'Stroma', 'tumor cells': 'Tumor cells', 'vasculature': 'Vasculature'}
CRC_MARKERS = ["aSMA - smooth muscle:Cyc_11_ch_2","beta-catenin - Wnt signaling:Cyc_4_ch_4","CD11b - macrophages:Cyc_10_ch_3",
 "CD11c - DCs:Cyc_12_ch_3","CD138 - plasma cells:Cyc_21_ch_3","CD15 - granulocytes:Cyc_14_ch_2","CD163 - macrophages:Cyc_17_ch_3",
 "CD20 - B cells:Cyc_8_ch_3","CD21 - DCs:Cyc_6_ch_4","CD3 - T cells:Cyc_16_ch_4","CD31 - vasculature:Cyc_19_ch_3",
 "CD34 - vasculature:Cyc_20_ch_3","CD38 - multifunctional:Cyc_20_ch_4","CD4 - T helper cells:Cyc_6_ch_3","CD44 - stroma:Cyc_2_ch_2",
 "CD45 - hematopoietic cells:Cyc_4_ch_2","CD45RA - naive T cells:Cyc_6_ch_2","CD45RO - memory cells:Cyc_18_ch_3",
 "CD56 - NK cells:Cyc_10_ch_4","CD68 - macrophages:Cyc_18_ch_4","CD7 - T cells:Cyc_16_ch_3","CD8 - cytotoxic T cells:Cyc_3_ch_2",
 "Chromogranin A - neuroendocrine:Cyc_17_ch_2","Cytokeratin - epithelia:Cyc_10_ch_2","FOXP3 - regulatory T cells:Cyc_2_ch_3",
 "GATA3 - Th2 helper T cells:Cyc_3_ch_4","GFAP - nerves:Cyc_16_ch_2","Granzyme B - cytotoxicity:Cyc_13_ch_2",
 "HLA-DR - MHC-II:Cyc_5_ch_2","MUC-1 - epithelia:Cyc_7_ch_2","Na-K-ATPase - membranes:Cyc_9_ch_2",
 "p53 - tumor suppressor:Cyc_3_ch_3","Podoplanin - lymphatics:Cyc_19_ch_4","Synaptophysin - neuroendocrine:Cyc_15_ch_3"]
PRESETS = {"authors_CRC_CODEX": dict(label_col="ClusterName", group_col="spots", area_col=False, marker_cols=CRC_MARKERS,
                                     class_mapping=CRC_CLASS_MAPPING, drop_unmapped=True)}

def find_col(df, cands, exclude=()):
    m = {norm(c): c for c in df.columns}
    for c in cands:
        if c in m and m[c] not in exclude: return m[c]
    return None

def list_files(path):
    paths = path if isinstance(path, (list, tuple)) else [path]
    files = []
    for p in paths:
        if os.path.isdir(p): files += [f for f in glob.glob(os.path.join(p, "**", "*"), recursive=True) if f.lower().endswith(EXTS)]
        else: files += glob.glob(p, recursive=True)
    files = sorted(set(files))
    if not files: raise FileNotFoundError(f"No data files found for: {path}")
    return files

def read_one(f, nrows=None):
    fl = f.lower()
    if fl.endswith(".h5ad"):
        a = anndata.read_h5ad(f)
        X = a.X.toarray() if hasattr(a.X, "toarray") else np.asarray(a.X)
        d = pd.DataFrame(X.astype(np.float32), columns=a.var_names.astype(str))
        obs = a.obs.reset_index(drop=True); obs.columns = [c + "_obs" if c in d.columns else c for c in obs.columns]
        d = pd.concat([d, obs], axis=1)
        return d.iloc[:nrows] if nrows else d
    if fl.endswith(".parquet"):
        d = pd.read_parquet(f); return d.iloc[:nrows] if nrows else d
    return pd.read_csv(f, sep="\t" if ".tsv" in fl else ",", nrows=nrows)

def load_table(path, nrows=None):
    files = list_files(path); parts = []
    for f in files:
        d = read_one(f, nrows); d["__file__"] = os.path.basename(f); parts.append(d)
        if nrows and len(parts) >= 3: break
    df = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
    for c in df.select_dtypes("float64").columns: df[c] = df[c].astype(np.float32)
    return df

def resolve_cfg(cfg):
    c = dict(PRESETS.get(cfg.get("preset"), {})); c.update({k: v for k, v in cfg.items() if v is not None}); return c

def detect(df, cfg):
    cfg = resolve_cfg(cfg)
    label_col = cfg.get("label_col") or find_col(df, LABEL_CAND)
    if label_col is None: raise ValueError(f"Could not find the cell-type column; set label_col. Columns: {list(df.columns)[:60]}")
    group_col = cfg.get("group_col") or find_col(df, GROUP_CAND, exclude=(label_col,))
    if group_col is None and "__file__" in df and df["__file__"].nunique() > 1: group_col = "__file__"
    if group_col is None: raise ValueError("No patient/sample/region column found; set group_col (MAPS needs groups to split by).")
    area = cfg.get("area_col", None)
    if area is None: area = find_col(df, AREA_CAND, exclude=(label_col, group_col))
    if area is False: area = None
    if cfg.get("marker_cols"):
        missing = [m for m in cfg["marker_cols"] if m not in df.columns]
        if missing: raise ValueError(f"marker_cols not in table: {missing[:5]} ...")
        markers = list(cfg["marker_cols"])
    else:
        skip = {label_col, group_col, area, "__file__"}
        markers = [c for c in df.select_dtypes("number").columns if c not in skip and not NONMARKER_RE.search(str(c))]
    markers = sorted(markers)                              # authors sort marker names, then append cell size last
    return label_col, group_col, area, markers

def authors_group_folds(group_counts, n_folds):
    """Authors' rule (CRC_CODEX.py): sort groups by size, pair smallest+largest, deal pairs to folds in turn.
    (With an odd number of groups the middle one stays unassigned = always in train, as in their code.)
    Their script used row positions instead of group names here, which crashes on text IDs; names are used here."""
    names = group_counts.sort_values(kind="stable").index.tolist()
    fold_of = {g: -1 for g in names}; i, j, fid = 0, len(names) - 1, 0
    while i < j:
        fold_of[names[i]] = fid % n_folds; fold_of[names[j]] = fid % n_folds
        fid += 1; i += 1; j -= 1
    return fold_of

def prepare(cfg, df):
    """-> features DataFrame (markers sorted + cell size), integer labels, group per cell, class names"""
    cfg = resolve_cfg(cfg)
    label_col, group_col, area, markers = detect(df, cfg)
    y = df[label_col].astype(str).str.strip()
    drop = {s.lower() for s in cfg.get("drop_labels", DROP_LABELS)}
    keep = ~y.str.lower().isin(drop)
    if cfg.get("class_mapping"):
        mp = cfg["class_mapping"]
        if cfg.get("drop_unmapped"): keep &= y.isin(mp.keys())
        y = y.map(lambda v: mp.get(v, v))
    cnt = y[keep].value_counts()
    small = cnt[cnt < MIN_CELLS_PER_CLASS].index.tolist()
    if small: print(f"  dropping {len(small)} class(es) with < {MIN_CELLS_PER_CLASS} cells: {small}")
    keep &= ~y.isin(small)
    feats = markers + ([area] if area else [])
    X = df.loc[keep, feats].astype(np.float32).fillna(0)
    const = [c for c in X.columns if X[c].nunique() <= 1]
    if const: print(f"  dropping constant columns: {const}"); X = X.drop(columns=const)
    y = y[keep].reset_index(drop=True); g = df.loc[keep, group_col].astype(str).reset_index(drop=True); X = X.reset_index(drop=True)
    class_names = sorted(y.unique().tolist()); cid = {c: i for i, c in enumerate(class_names)}
    labels = y.map(cid).values
    return X, labels, g, class_names, dict(label_col=label_col, group_col=group_col, area_col=area, n_markers=len(markers) - len(const) + (0 if not area or area not in const else 0))

def class_counts_by_fold(labels, g, fold_of, k):
    f = g.map(fold_of).values
    return {i: np.bincount(labels[f == i], minlength=k) for i in range(-1, max(fold_of.values()) + 1)}

def _header(f):
    fl = f.lower()
    if fl.endswith(".h5ad"): return ("__h5ad__", f)            # each h5ad is its own group
    if fl.endswith(".parquet"):
        import pyarrow.parquet as pq; return tuple(pq.read_schema(f).names)
    return tuple(pd.read_csv(f, sep="\t" if ".tsv" in fl else ",", nrows=0).columns)

def discover(root):
    files = [f for f in glob.glob(os.path.join(root, "**", "*"), recursive=True) if f.lower().endswith(EXTS)]
    if not files: raise FileNotFoundError(f"No csv/tsv/parquet/h5ad files found under {root}. Did you attach the dataset?")
    groups = {}
    for f in files:
        try: groups.setdefault(_header(f), []).append(f)
        except Exception as e: print("  cannot read header of", f, "->", e)
    best, best_key = None, None
    print(f"Found {len(files)} table file(s) in {len(groups)} column-signature group(s):")
    for hdr, fs in groups.items():
        if hdr and hdr[0] == "__h5ad__":
            try:
                a = anndata.read_h5ad(fs[0], backed="r"); cols = list(a.obs.columns); a.file.close()
            except Exception: cols = []
        else: cols = list(hdr)
        has_label = find_col(pd.DataFrame(columns=cols), LABEL_CAND) is not None
        size = sum(os.path.getsize(f) for f in fs)
        print(f"  {len(fs):4d} file(s) | {size/1e6:9.1f} MB | cell-type column: {'yes' if has_label else 'no '} | e.g. {os.path.basename(fs[0])}")
        key = (has_label, size)
        if best_key is None or key > best_key: best, best_key = fs, key
    if not best_key[0]: print("  WARNING: no group has an obvious cell-type column; picking the largest. Set OVERRIDES if wrong.")
    return sorted(best)

def auto_preset(df):
    need = set(PRESETS["authors_CRC_CODEX"]["marker_cols"]) | {"ClusterName", "spots"}
    return "authors_CRC_CODEX" if need <= set(df.columns) else None

def metrics_one_fold(gt, pred, k):
    p, r, f, s = precision_recall_fscore_support(gt, pred, labels=list(range(k)), zero_division=0)
    present = s > 0
    micro_f1 = (np.asarray(gt) == np.asarray(pred)).mean()   # authors' "Average" bar is a micro average (= accuracy)
    return p, r, f, s, present, micro_f1


## 7. Check (fast): what was detected for each dataset?

In [ ]:
files = discover(INPUT_ROOT)
name = re.sub(r"[^A-Za-z0-9_.-]+", "_", os.path.basename(os.path.commonpath([os.path.dirname(f) for f in files])) or "dataset")
cfg0 = dict(path=files, **OVERRIDES)
if "preset" not in cfg0:
    p = auto_preset(read_one(files[0], nrows=5))
    if p: cfg0["preset"] = p; print("Schurch CRC CODEX table recognised -> using the authors' own label mapping and markers")
DATASETS = {name: cfg0}
print("Dataset name:", name, "| files used:", len(files))

def selected():
    return list(DATASETS)

for name in selected():
    cfg = DATASETS[name]; print("=" * 80, "\n", name)
    try:
        df = load_table(cfg["path"], nrows=5000)
        lab, grp, area, mk = detect(df, cfg)
        print(f"  files: {len(list_files(cfg['path']))} | label_col: {lab} | group_col: {grp} ({df[grp].nunique()} groups in first rows) | cell size col: {area}")
        print(f"  classes (first rows): {df[lab].astype(str).nunique()} | markers ({len(mk)}): {mk[:40]}")
        print(f"  all columns: {list(df.columns)[:60]}")
    except Exception as e:
        print("  ERROR:", e)

## 8. Main run
One dataset at a time. For each fold: write temporary train/valid/test CSVs → authors' `Trainer.fit` → authors' `Predictor.predict` on the **test fold** → metrics → delete the temporary CSVs. Finished folds are skipped if you re-run.

In [ ]:
def run_dataset(name, cfg):
    t0 = time.time(); print("=" * 80, "\n", name)
    df = load_table(cfg["path"])
    X, labels, g, class_names, info = prepare(cfg, df)
    del df; gc.collect()
    k = len(class_names); nfeat = X.shape[1]
    hp = dict(DEFAULT_HP); hp.update(cfg.get("hp", {}))
    n_groups = g.nunique()
    n_folds = min(N_FOLDS, n_groups // 2)
    if n_folds < 2: raise ValueError(f"Only {n_groups} groups in '{info['group_col']}'; need >= 4. Use a finer group_col (e.g. region/FOV), but that can leak patients.")
    if n_folds < N_FOLDS: print(f"  WARNING: only {n_groups} groups -> using {n_folds} folds")
    fold_of = authors_group_folds(g.value_counts(), n_folds)
    cc = class_counts_by_fold(labels, g, fold_of, k)
    print(f"  cells: {len(labels):,} | classes: {k} | features: {nfeat} | groups ({info['group_col']}): {n_groups} | folds: {n_folds}")
    print(f"  hp: {hp}")

    out = os.path.join(OUT_ROOT, name); os.makedirs(out, exist_ok=True)
    pd.DataFrame({"class_id": range(k), "class_name": class_names, "cells": np.bincount(labels, minlength=k)}).to_csv(f"{out}/class_names.csv", index=False)
    json.dump(dict(maps_commit="72d4bc81cf4e28f19a766693c885670e47ab3e6e", features=X.columns.tolist(), hp=hp, folds=n_folds, group_col=info["group_col"],
                   patches="np.Inf->np.inf; torch.load weights_only=False"), open(f"{out}/run_info.json", "w"), indent=1)

    fold_arr = g.map(fold_of).values
    rows, cm_total, summaries = [], np.zeros((k, k)), []
    for fold in range(n_folds):
        res_path = f"{out}/fold_{fold}/results_test.csv"
        if os.path.exists(res_path):
            r = pd.read_csv(res_path); gt, pred = r["gt_label"].values, r["pred_label"].values
            print(f"  fold {fold}: found saved results, skipping training")
        else:
            test_idx = np.where(fold_arr == fold)[0]; trainval_idx = np.where(fold_arr != fold)[0]
            # authors' cHL_CODEX.py: 80/20 stratified split of the training part -> train / valid (first split only)
            skf = StratifiedKFold(n_splits=5, random_state=SPLIT_SEED, shuffle=True)
            tr_i, va_i = next(iter(skf.split(np.zeros(len(trainval_idx)), labels[trainval_idx])))
            tr_idx, va_idx = trainval_idx[tr_i], trainval_idx[va_i]
            miss = [class_names[c] for c in range(k) if (np.bincount(labels[va_idx], minlength=k)[c] == 0) or (np.bincount(labels[tr_idx], minlength=k)[c] == 0)]
            if miss: raise ValueError(f"fold {fold}: class(es) {miss} missing from train or valid; raise MIN_CELLS_PER_CLASS or merge classes (class_mapping).")
            tmp = f"{out}/fold_{fold}/tmp"; os.makedirs(tmp, exist_ok=True)
            for nm, idx in [("train", tr_idx), ("valid", va_idx), ("test", test_idx)]:
                d = X.iloc[idx].copy(); d["cell_label"] = labels[idx]; d.to_csv(f"{tmp}/{nm}.csv", index=False, float_format="%.8g")
            ck_dir = f"{out}/fold_{fold}"
            tf = time.time()
            model = Trainer(results_dir=ck_dir, num_features=nfeat, num_classes=k, batch_size=hp["batch_size"],
                            learning_rate=hp["learning_rate"], dropout=hp["dropout"], max_epochs=hp["max_epochs"],
                            min_epochs=hp["min_epochs"], patience=hp["patience"], seed=hp["seed"],
                            num_workers=hp["num_workers"], verbose=0)
            with open(f"{ck_dir}/train_stdout.log", "w") as fh, contextlib.redirect_stdout(fh):   # their per-epoch prints go to a file
                model.fit(f"{tmp}/train.csv", f"{tmp}/valid.csv")
            pr = Predictor(model_checkpoint_path=f"{ck_dir}/best_checkpoint.pt", num_features=nfeat, num_classes=k,
                           batch_size=hp["batch_size"], seed=hp["seed"], num_workers=hp["num_workers"], normalization=True, verbose=0)
            with contextlib.redirect_stdout(open(os.devnull, "w")):
                pred, probs = pr.predict(f"{tmp}/test.csv")
            gt = labels[test_idx]
            r = pd.DataFrame({"pred_label": pred, "gt_label": gt, "group": g.values[test_idx]})
            if SAVE_PROBS: r = pd.concat([r, pd.DataFrame(probs, columns=[c + "_prob" for c in class_names])], axis=1)
            r.to_csv(res_path, index=False)
            shutil.rmtree(tmp, ignore_errors=True)
            lg = pd.read_csv(f"{ck_dir}/training_logs.csv")
            print(f"  fold {fold}: {len(lg)} epochs, best valid loss {lg['valid_loss'].min():.4f}, {(time.time()-tf)/60:.1f} min")
        p, rc, f1, s, present, micro = metrics_one_fold(gt, pred, k)
        for c in range(k):
            rows.append(dict(fold=fold, cell_type=class_names[c], precision=p[c] if present[c] else np.nan,
                             recall=rc[c] if present[c] else np.nan, f1=f1[c] if present[c] else np.nan, support=int(s[c])))
        summaries.append(dict(fold=fold, macro_f1=np.nanmean(f1[present]), micro_f1=micro))
        cm_total += confusion_matrix(gt, pred, labels=list(range(k)))
        print(f"           macro F1 = {summaries[-1]['macro_f1']:.3f} | micro F1 (= accuracy) = {micro:.3f}")

    res = pd.DataFrame(rows); sm = pd.DataFrame(summaries)
    res.to_csv(f"{out}/f1_per_fold_per_class.csv", index=False); sm.to_csv(f"{out}/f1_per_fold_overall.csv", index=False)
    table = res.groupby("cell_type")[["precision", "recall", "f1"]].agg(["mean", "std"]).round(3)
    table.to_csv(f"{out}/f1_summary_per_class.csv"); print(table)
    print(f"  AVERAGE F1 over cell types (macro): {sm.macro_f1.mean():.3f} ± {sm.macro_f1.std():.3f} | micro: {sm.micro_f1.mean():.3f} ± {sm.micro_f1.std():.3f}")

    f1m = res.groupby("cell_type")["f1"].agg(["mean", "std"]).sort_values("mean")
    fig, ax = plt.subplots(1, 2, figsize=(14, max(4, 0.35 * k + 1.5)))
    ax[0].barh(f1m.index, f1m["mean"], xerr=f1m["std"].fillna(0), color="steelblue"); ax[0].set_xlim(0, 1)
    ax[0].set_xlabel("F1 (mean ± SD across folds)"); ax[0].set_title(name)
    cmn = cm_total / np.clip(cm_total.sum(1, keepdims=True), 1, None)
    sns.heatmap(cmn, annot=k <= 15, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax[1])
    ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Ground truth")
    plt.tight_layout(); plt.savefig(f"{out}/f1_and_confusion.png", dpi=150); plt.show(); plt.close()
    return dict(dataset=name, cells=len(labels), cell_types=k, groups=n_groups, folds=n_folds, features=nfeat,
                macro_f1=round(sm.macro_f1.mean(), 3), macro_sd=round(sm.macro_f1.std(), 3),
                micro_f1=round(sm.micro_f1.mean(), 3), minutes=round((time.time() - t0) / 60, 1), error="")

ALL = []
for name in selected():
    try: ALL.append(run_dataset(name, DATASETS[name]))
    except Exception as e:
        import traceback; traceback.print_exc(); ALL.append(dict(dataset=name, error=repr(e)))
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 9. Combined summary (datasets run in this session)

In [ ]:
summary = pd.DataFrame(ALL); display(summary)
summary.to_csv(f"{OUT_ROOT}/summary_this_session.csv", index=False)